# Phase 9: Leaf Detector YOLOv8 Training Loop

This notebook trains a custom **YOLOv8n object detection model** on the PlantDoc dataset to detect and locate crop leaves.

It performs **class collapsing** to map all 30 disease/crop-specific classes into a single generic **`leaf`** class (ID `0`), which is used by our backend gateway validation pipeline.

In [ ]:
# Setup paths and imports
from pathlib import Path
import sys
import os
import yaml
import shutil
import torch
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'ai' and PROJECT_ROOT.parent != PROJECT_ROOT:
    if (PROJECT_ROOT / 'ai').exists():
        PROJECT_ROOT = PROJECT_ROOT / 'ai'
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import utils

### 1. Perform Class Collapsing on Dataset

In [5]:
dataset_dir = utils.paths.DETECTION_DATASET_DIR
original_yaml_path = dataset_dir / "data.yaml"
collapsed_yaml_path = dataset_dir / "data_collapsed.yaml"

if original_yaml_path.exists():
    with open(original_yaml_path, 'r') as f:
        data = yaml.safe_load(f)
        
    # Generate single class collapsed config
    data_collapsed = {
        'path': str(dataset_dir.resolve()).replace('\\', '/'),
        'train': 'train/images',
        'val': 'test/images',  # Fallback validation to test images split
        'test': 'test/images',
        'names': {0: 'leaf'},
        'nc': 1
    }
    
    with open(collapsed_yaml_path, 'w') as f:
        yaml.safe_dump(data_collapsed, f, default_flow_style=False)
    print("✅ data_collapsed.yaml created at:", collapsed_yaml_path)
else:
    print("⚠️ Warning: Dataset config data.yaml not found on disk.")

def collapse_labels_inplace(split_name):
    labels_dir = dataset_dir / split_name / "labels"
    backup_dir = dataset_dir / split_name / "labels_original"
    
    if labels_dir.exists():
        # Create folder backup for safety
        if not backup_dir.exists():
            print(f"Backing up original labels for {split_name} to {backup_dir.name}...")
            shutil.copytree(labels_dir, backup_dir)
            
        label_files = list(labels_dir.glob("*.txt"))
        print(f"Collapsing {len(label_files)} label files in-place for {split_name}...")
        
        for fpath in label_files:
            with open(fpath, 'r') as f:
                lines = f.readlines()
            
            collapsed_lines = []
            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 5:
                    parts[0] = '0'  # Class ID forced to 0 (leaf)
                    collapsed_lines.append(" ".join(parts))
            
            with open(fpath, 'w') as f:
                f.write("\n".join(collapsed_lines) + "\n")
    else:
        print(f"⚠️ Labels folder not found for {split_name}.")

collapse_labels_inplace("train")
collapse_labels_inplace("test")
print("✅ Labels collapse pre-processing completed successfully.")

✅ data_collapsed.yaml created at: O:\Hackthons\KrishiOS\ai\datasets\detection\plantdoc\data_collapsed.yaml
Collapsing 2328 label files in-place for train...
Collapsing 239 label files in-place for test...
✅ Labels collapse pre-processing completed successfully.


### 2. Execute YOLOv8 Training Loop

In [6]:
# Redirect model weights and training logs to outputs folder
from ultralytics import settings
settings.update({
    'weights_dir': str(utils.paths.DETECTOR_MODEL_DIR.resolve()),
    'runs_dir': str(utils.paths.PROJECT_ROOT.resolve() / "outputs" / "runs")
})

# Load base model weights from detector models folder
yolo_base = utils.paths.DETECTOR_MODEL_DIR / "yolov8n.pt"
model = YOLO(str(yolo_base))

print("🚀 Starting YOLO Training...")
results = model.train(
    data=str(collapsed_yaml_path.resolve()).replace('\\', '/'),
    epochs=15,
    imgsz=416,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    name="yolov8n_leaf_detection"
)

🚀 Starting YOLO Training...
Ultralytics 8.4.102  Python-3.13.9 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=O:/Hackthons/KrishiOS/ai/datasets/detection/plantdoc/data_collapsed.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=O:\Hackthons\KrishiOS\ai\models\detector\yolov8n.pt